# Born-projected angular coefficients from POWHEG LHE events

This notebook builds the first complete analysis chain for off-shell $H\to ZZ\to e^+e^-\mu^+\mu^-$ events:

1. read complete signal, background, and signal+background+interference LHE samples with `pylhe`;
2. select only the final-state $e^-$, $e^+$, $\mu^-$, and $\mu^+$ and reconstruct $Z_1$, $Z_2$, and the Higgs candidate from their four-vector sums;
3. remove the color-singlet transverse recoil with the Born projection of [arXiv:2606.11083](https://arxiv.org/abs/2606.11083);
4. construct the angular coordinates and standard four-lepton observables, following the terminology of [arXiv:1208.4018](https://arxiv.org/abs/1208.4018);
5. project the full inclusive symmetric basis through $\ell_{\max}=3$, including signed-weight MC uncertainties and significances;
6. compare the differential $S_{00;20}$ and $S_{20;20}$ coefficients among the three samples as functions of $m_{ZZ}$ and $\cos\theta^*$;
7. train reweighted density-ratio estimators for both coefficients with [`nsbi-common-utils`](https://github.com/iris-hep/nsbi-lhc-toolkit); and
8. compare each learned differential coefficient with its direct weighted projection.

Our fixed off-shell convention is $Z_1=Z_{\mu\mu}$ and $Z_2=Z_{ee}$; there is no mass ordering. The harmonic coordinates use the **positively charged** lepton in each $Z$ rest frame. The separately named `theta1_standard` and `theta2_standard` use the negatively charged leptons, as in arXiv:1208.4018.

In [ ]:
from pathlib import Path
import sys
import warnings

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import matplotlib.pyplot as plt
import mplhep as hep
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from offshell_angles import (
    angular_modes,
    as_float32_features,
    binned_angular_coefficient,
    class_balanced_validation_bce,
    inclusive_angular_statistics,
    angular_target,
    load_lhe_dataframe,
    prepare_weighted_classification,
    recover_conditional_moment,
    validation_loss_outlier_mask,
)

hep.style.use("ATLAS")
pd.set_option("display.max_columns", 80)
RNG_SEED = 20260718

## 0. Runtime device check

Run this before loading the LHE sample. For CPU execution the path should contain `.pixi/envs/analysis/` and CUDA availability should be `False`. For GPU execution the path should contain `.pixi/envs/analysis-gpu/`, the compiled CUDA version should be non-`None`, and CUDA availability should be `True`.

In [ ]:
import torch

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("Compiled CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 1. Read the three complete local LHE samples

Set the three paths in `SAMPLE_CONFIG` to files visible inside the JupyterLab pod. Both plain `.lhe` and gzip-compressed `.lhe.gz` files are supported by `pylhe`. The analysis deliberately calls the reader with `max_events=None`: every event is read so that signed-weight normalizations, inclusive coefficients, and differential bin sums refer to the complete generated samples.

With `STRICT_TOPOLOGY=True`, the reader stops unless every accepted event has exactly one final-state $e^-$, $e^+$, $\mu^-$, and $\mu^+$. Set it to `False` only when intentionally filtering a mixed-topology LHE file.

In [ ]:
SAMPLE_CONFIG = {
    "signal": {"label": "S", "path": Path("../data/signal.lhe"), "color": "C3"},
    "background": {"label": "B", "path": Path("../data/background.lhe"), "color": "C0"},
    "signal_background_interference": {
        "label": "S+B+I",
        "path": Path("../data/signal_background_interference.lhe"),
        "color": "C2",
    },
}
STRICT_TOPOLOGY = True
PRIMARY_SAMPLE = "signal_background_interference"

missing_files = [
    f"{key}: {config['path']}"
    for key, config in SAMPLE_CONFIG.items()
    if not config["path"].is_file()
]
if missing_files:
    raise FileNotFoundError("Update SAMPLE_CONFIG paths:\n" + "\n".join(missing_files))

samples = {}
for sample_key, config in SAMPLE_CONFIG.items():
    samples[sample_key] = load_lhe_dataframe(
        config["path"],
        max_events=None,
        strict=STRICT_TOPOLOGY,
        include_momenta=False,
    )
    print(f"Loaded {len(samples[sample_key]):,} selected {config['label']} events")

# The density-ratio demonstration below trains on this sample, while all
# three samples enter the inclusive and direct differential comparisons.
events = samples[PRIMARY_SAMPLE]
samples[PRIMARY_SAMPLE].head()

In [ ]:
sample_summary = pd.DataFrame.from_dict(
    {
        sample_key: {
            "label": SAMPLE_CONFIG[sample_key]["label"],
            "events": len(sample_events),
            "sum_weights": sample_events["weight"].sum(),
            "sqrt_sum_weights_squared": np.sqrt(np.sum(sample_events["weight"] ** 2)),
            "negative_weight_fraction": (sample_events["weight"] < 0).mean(),
            "degenerate_local_frames": sample_events["frame_degenerate"].sum(),
            "degenerate_standard_angles": sample_events["standard_angles_degenerate"].sum(),
        }
        for sample_key, sample_events in samples.items()
    },
    orient="index",
)
sample_summary

## 2. Born-like projection

Let $k=\sum_{i=1}^4p_{\ell_i}$ be the measured four-lepton momentum. The projection applies the same Lorentz transformation to every lepton:

$$p_i^{\mathrm B}=B_L^{-1}B_TB_Lp_i.$$

$B_L$ first removes the longitudinal momentum of $k$, $B_T$ then removes the transverse momentum of the longitudinally boosted system, and $B_L^{-1}$ restores the original color-singlet rapidity. Consequently,

$$m_{4\ell}^{\mathrm B}=m_{4\ell},\qquad y_{4\ell}^{\mathrm B}=y_{4\ell},\qquad p_{T,4\ell}^{\mathrm B}=0,$$

and every internal invariant mass, including $m_{\mu\mu}$ and $m_{ee}$, is unchanged. The following checks should be at floating-point precision.

In [ ]:
projection_rows = []
for sample_key, sample_events in samples.items():
    delta_m4l = sample_events["born_m4l"] - sample_events["raw_m4l"]
    delta_y4l = sample_events["born_y4l"] - sample_events["raw_y4l"]
    relative_pt4l = sample_events["born_pt4l"] / np.maximum(sample_events["raw_m4l"], 1.0)
    projection_rows.append(
        {
            "sample": SAMPLE_CONFIG[sample_key]["label"],
            "max_abs_delta_m4l": np.max(np.abs(delta_m4l)),
            "max_abs_delta_y4l": np.max(np.abs(delta_y4l)),
            "max_born_pt4l_over_m4l": np.max(relative_pt4l),
        }
    )
    assert np.allclose(sample_events["born_m4l"], sample_events["raw_m4l"], rtol=2e-10, atol=2e-10)
    assert np.allclose(sample_events["born_y4l"], sample_events["raw_y4l"], rtol=2e-10, atol=2e-10)
    assert np.max(relative_pt4l) < 2e-10

pd.DataFrame(projection_rows)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hist(events["raw_pt4l"], bins=60, weights=events["weight"], histtype="step", label="POWHEG")
axes[0].hist(events["born_pt4l"], bins=60, weights=events["weight"], histtype="step", label="Born projected")
axes[0].set(xlabel=r"$p_{T,4\ell}$ [GeV]", ylabel="weighted events")
axes[0].legend()
axes[1].scatter(events["raw_y4l"], events["born_y4l"] - events["raw_y4l"], s=4, alpha=0.25)
axes[1].set(xlabel=r"raw $y_{4\ell}$", ylabel=r"$y_{4\ell}^{B}-y_{4\ell}$")
#fig.tight_layout()

## 3. Angular and differential variables

In the Born-projected four-lepton rest frame, the local $\hat z_i$ axis follows $Z_i$. The $\hat y_i$ axis is normal to the beam--$Z_i$ production plane and $\hat x_i=\hat y_i\times\hat z_i$. After boosting the positively charged lepton into its parent-$Z$ rest frame, its direction defines $\Omega_i=(\theta_i,\phi_i)$.

The reader also supplies the usual variables $m_{Z_1}$, $m_{Z_2}$, $m_{ZZ}$, $\cos\theta^*$, $\Phi$, $\Phi_1$, and $\Psi=\Phi_1+\Phi/2$ (wrapped to $[-\pi,\pi)$). Here $Z_1$ is always the dimuon system. At exactly collinear production, an azimuthal origin is mathematically undefined; the record flags this measure-zero case.

In [ ]:
angle_columns = ["cos_theta1", "phi1", "cos_theta2", "phi2", "cos_theta_star", "Phi"]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for axis, column in zip(axes.flat, angle_columns):
    axis.hist(events[column].dropna(), bins=50, weights=events.loc[events[column].notna(), "weight"], histtype="step")
    axis.set(xlabel=column, ylabel="weighted events")
fig.tight_layout()

events[["m_Z1", "m_Z2", "m_ZZ", "cos_theta_star", "frame_degenerate", "standard_angles_degenerate"]].describe(include="all")

## 4. Harmonic projection and the $4\pi$ convention

We use

$$p(\Omega_1,\Omega_2,x)=\frac{1}{4\pi}\sum_{\alpha\preceq\beta}\mathcal S_{\alpha\beta}(x)\,\mathcal Y^{(+)}_{\alpha\beta}(\Omega_1,\Omega_2).$$

For an orthonormal basis, multiplication by $\mathcal Y^{(+)*}_{\alpha\beta}$ and integration over both solid angles gives

$$\mathcal S_{\alpha\beta}(x)=4\pi\int d\Omega_1d\Omega_2\;\mathcal Y^{(+)*}_{\alpha\beta}(\Omega_1,\Omega_2)\,p(\Omega_1,\Omega_2,x).$$

In particular, $\mathcal Y^{(+)}_{00;00}=1/(4\pi)$, so angular integration returns $\mathcal S_{00;00}(x)$. A direct MC estimator in an $x$ bin is therefore

$$\widehat{\mathcal S}_{\alpha\beta}(\text{bin})=4\pi\sum_{i\in\text{bin}}w_i\,\mathcal Y^{(+)*}_{\alpha\beta}(\Omega_{1,i},\Omega_{2,i}).$$

For complex harmonics we train the real and imaginary components separately.

In [ ]:
COEFFICIENTS = {
    "00;20": dict(l1=0, m1=0, l2=2, m2=0, component="real"),
    "20;20": dict(l1=2, m1=0, l2=2, m2=0, component="real"),
}

def coefficient_slug(coefficient_key):
    return coefficient_key.replace(";", "_")

coefficient_metadata = {}
direct_inclusive_rows = []
for sample_key, sample_events in samples.items():
    for coefficient_key, definition in COEFFICIENTS.items():
        slug = coefficient_slug(coefficient_key)
        h, target, bound = angular_target(
            sample_events["theta1"].to_numpy(),
            sample_events["phi1"].to_numpy(),
            sample_events["theta2"].to_numpy(),
            sample_events["phi2"].to_numpy(),
            **definition,
        )
        sample_events[f"h_{slug}"] = h
        sample_events[f"t_{slug}"] = target
        coefficient_metadata[coefficient_key] = {"bound": bound, **definition}

        valid = (
            np.isfinite(sample_events["weight"].to_numpy())
            & np.isfinite(h)
            & np.isfinite(target)
        )
        contributions = 4.0 * np.pi * sample_events.loc[valid, "weight"].to_numpy() * h[valid]
        direct_inclusive_rows.append(
            {
                "sample": SAMPLE_CONFIG[sample_key]["label"],
                "coefficient": coefficient_key,
                "events": np.count_nonzero(valid),
                "valid_fraction": np.mean(valid),
                "S": np.sum(contributions),
                "sigma_MC": np.sqrt(np.sum(contributions**2)),
                "harmonic_bound": bound,
            }
        )

direct_selected_coefficients = pd.DataFrame(direct_inclusive_rows)
direct_selected_coefficients

## 5. Inclusive $S_{\alpha\beta}$ through $\ell_{\max}=3$

Write $\alpha=(\ell_1,m_1)$ and $\beta=(\ell_2,m_2)$. We retain each symmetric basis element exactly once using

$$\alpha\preceq\beta\quad\Longleftrightarrow\quad \ell_1<\ell_2\quad\text{or}\quad(\ell_1=\ell_2\ \text{and}\ m_1\leq m_2).$$

For $0\leq\ell\leq3$ there are $\sum_{\ell=0}^{3}(2\ell+1)=16$ one-particle modes and $16\times17/2=136$ unordered mode pairs. No neural network is needed for the inclusive projection:

$$\widehat S_{\alpha\beta}=4\pi\sum_i w_i\,\mathcal Y^{(+)*}_{\alpha\beta}(\Omega_{1,i},\Omega_{2,i}).$$

For each sample, the table below retains the raw complex coefficient, its value relative to $S_{00;00}$, and the corresponding MC uncertainty and significance. For a component $A=\operatorname{Re}S_{\alpha\beta}$ or $\operatorname{Im}S_{\alpha\beta}$ and $B=S_{00;00}$, the weighted-sum estimates are $\operatorname{Var}(A)=\sum_i a_i^2$, $\operatorname{Var}(B)=\sum_i w_i^2$, and $\operatorname{Cov}(A,B)=\sum_i a_iw_i$, where $a_i=4\pi w_i\operatorname{Re/Im}\mathcal Y^{(+)*}_{\alpha\beta,i}$. The ratio uncertainty uses the correlated delta method,

$$\operatorname{Var}(A/B)=\frac{\operatorname{Var}(A)+(A/B)^2\operatorname{Var}(B)-2(A/B)\operatorname{Cov}(A,B)}{B^2}.$$

All central values use the original signed weights. Squaring occurs only in the MC-variance estimator. The four maps for each sample show the real and imaginary parts of $S_{\alpha\beta}/S_{00;00}$ and their signed significances; the redundant half of each matrix is masked.

In [ ]:
from matplotlib.colors import SymLogNorm, TwoSlopeNorm

INCLUSIVE_LMAX = 3

def inclusive_statistics_table(statistics):
    rows = []
    for alpha_index, (l1, m1) in enumerate(statistics.modes):
        for beta_index in range(alpha_index, len(statistics.modes)):
            l2, m2 = statistics.modes[beta_index]
            coefficient = statistics.coefficients[alpha_index, beta_index]
            relative = statistics.relative_coefficients[alpha_index, beta_index]
            sigma_real = statistics.relative_uncertainties_real[alpha_index, beta_index]
            sigma_imag = statistics.relative_uncertainties_imag[alpha_index, beta_index]
            significance_real = statistics.significances_real[alpha_index, beta_index]
            significance_imag = statistics.significances_imag[alpha_index, beta_index]
            rows.append(
                {
                    "alpha_index": alpha_index,
                    "beta_index": beta_index,
                    "l1": l1,
                    "m1": m1,
                    "l2": l2,
                    "m2": m2,
                    "alpha": f"({l1},{m1})",
                    "beta": f"({l2},{m2})",
                    "S": coefficient,
                    "Re(S/S0000)": relative.real,
                    "Im(S/S0000)": relative.imag,
                    "sigma_Re(S/S0000)": sigma_real,
                    "sigma_Im(S/S0000)": sigma_imag,
                    "significance_Re": significance_real,
                    "significance_Im": significance_imag,
                    "max_abs_significance": np.nanmax(
                        np.abs([significance_real, significance_imag])
                    ) if np.any(np.isfinite([significance_real, significance_imag])) else np.nan,
                    "valid_fraction": statistics.valid_fractions[alpha_index, beta_index],
                }
            )
    return pd.DataFrame(rows)


def draw_inclusive_map(axis, values, title, *, significance=False):
    masked = np.ma.masked_invalid(values.T)
    finite_values = masked.compressed()
    max_abs = (
        float(np.max(np.abs(finite_values)))
        if finite_values.size
        else np.finfo(float).eps
    )
    max_abs = max(max_abs, np.finfo(float).eps)
    if significance:
        norm = TwoSlopeNorm(vmin=-max_abs, vcenter=0.0, vmax=max_abs)
    else:
        norm = SymLogNorm(
            linthresh=max_abs * 1.0e-3,
            linscale=1.0,
            vmin=-max_abs,
            vmax=max_abs,
            base=10,
        )
    image = axis.imshow(masked, origin="upper", cmap="RdBu_r", norm=norm, aspect="equal")
    return image


inclusive_results = {}
inclusive_tables = {}
expected_mode_count = (INCLUSIVE_LMAX + 1) ** 2
expected_coefficient_count = expected_mode_count * (expected_mode_count + 1) // 2
mode_labels = [f"({ell},{m})" for ell, m in angular_modes(INCLUSIVE_LMAX)]

for sample_key, sample_events in samples.items():
    label = SAMPLE_CONFIG[sample_key]["label"]
    statistics = inclusive_angular_statistics(
        sample_events["theta1"].to_numpy(),
        sample_events["phi1"].to_numpy(),
        sample_events["theta2"].to_numpy(),
        sample_events["phi2"].to_numpy(),
        sample_events["weight"].to_numpy(),
        l_max=INCLUSIVE_LMAX,
    )
    inclusive_results[sample_key] = statistics
    table = inclusive_statistics_table(statistics)
    table.insert(0, "sample", label)
    inclusive_tables[sample_key] = table

    np.testing.assert_allclose(
        statistics.coefficients[0, 0],
        sample_events.loc[np.isfinite(sample_events["weight"]), "weight"].sum(),
        rtol=1.0e-12,
        atol=1.0e-10,
    )
    assert len(table) == expected_coefficient_count

    display(
        table.sort_values("max_abs_significance", ascending=False)
        .head(25)
        .reset_index(drop=True)
    )

    fig, axes = plt.subplots(2, 2, figsize=(17, 14), constrained_layout=True)
    map_specs = (
        (statistics.relative_coefficients.real, r"$\mathrm{Re}(S_{\alpha\beta}/S_{00;00})$", False),
        (statistics.relative_coefficients.imag, r"$\mathrm{Im}(S_{\alpha\beta}/S_{00;00})$", False),
        (statistics.significances_real, r"$\mathrm{Re}(S/S_{00;00})/\sigma_{\mathrm{Re}}$", True),
        (statistics.significances_imag, r"$\mathrm{Im}(S/S_{00;00})/\sigma_{\mathrm{Im}}$", True),
    )
    for axis, (values, title, significance) in zip(axes.flat, map_specs):
        image = draw_inclusive_map(axis, values, title, significance=significance)
        axis.set_xticks(range(len(mode_labels)), labels=mode_labels, rotation=90)
        axis.set_yticks(range(len(mode_labels)), labels=mode_labels)
        axis.set_xlabel(r"$\alpha=(\ell_1,m_1)$")
        axis.set_ylabel(r"$\beta=(\ell_2,m_2)$")
        axis.set_title(title)
        fig.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
    fig.suptitle(
        rf"{label}: inclusive symmetric coefficients, $\ell_{{\max}}={INCLUSIVE_LMAX}$"
    )
    plt.show()

inclusive_coefficients = pd.concat(inclusive_tables, names=["sample_key", "row"])
print(f"Stored {expected_coefficient_count} inclusive coefficients for each of {len(samples)} samples")


## 6. Direct differential comparisons for $S_{00;20}$ and $S_{20;20}$

For each selected real coefficient and observable $x$, the signed-weight bin estimator is

$$
\widehat S_{\alpha\beta,b}=4\pi\sum_{i\in b}w_i h_{\alpha\beta,i},
\qquad
\widehat\sigma_{\alpha\beta,b}=
\sqrt{\sum_{i\in b}\left(4\pi w_i h_{\alpha\beta,i}\right)^2}.
$$

The central values retain every positive or negative event weight exactly as stored in the LHE file. No division by $S_{00;00}$ is applied to these histograms, so their normalization preserves the total signed cross-section information. The same bin edges are used for all three samples. Finite underflow and overflow events are folded into the edge bins, ensuring that the sum over bins reproduces the corresponding inclusive coefficient even if a custom plotting range is chosen.


In [ ]:
finite_mzz = np.concatenate([
    sample_events.loc[np.isfinite(sample_events["m_ZZ"]), "m_ZZ"].to_numpy()
    for sample_events in samples.values()
])
MZZ_RANGE = (float(np.min(finite_mzz)), float(np.max(finite_mzz)))
DIFFERENTIAL_BINS = {
    "m_ZZ": np.linspace(*MZZ_RANGE, 25),
    "cos_theta_star": np.linspace(-1.0, 1.0, 21),
}
OBSERVABLE_LABELS = {
    "m_ZZ": r"$m_{ZZ}$ [GeV]",
    "cos_theta_star": r"$\cos\theta^*$",
}

differential_results = {}
fig, axes = plt.subplots(
    len(COEFFICIENTS),
    len(DIFFERENTIAL_BINS),
    figsize=(15, 10),
    constrained_layout=True,
    squeeze=False,
)

for row_index, coefficient_key in enumerate(COEFFICIENTS):
    slug = coefficient_slug(coefficient_key)
    differential_results[coefficient_key] = {}
    for column_index, (observable, bin_edges) in enumerate(DIFFERENTIAL_BINS.items()):
        axis = axes[row_index, column_index]
        differential_results[coefficient_key][observable] = {}
        for sample_key, sample_events in samples.items():
            config = SAMPLE_CONFIG[sample_key]
            result = binned_angular_coefficient(
                sample_events[observable].to_numpy(),
                sample_events[f"h_{slug}"].to_numpy(),
                sample_events["weight"].to_numpy(),
                bin_edges,
                fold_flow=True,
            )
            differential_results[coefficient_key][observable][sample_key] = result
            axis.stairs(
                result.values,
                result.bin_edges,
                color=config["color"],
                linewidth=1.8,
                label=config["label"],
            )
            lower = result.values - result.uncertainties
            upper = result.values + result.uncertainties
            axis.fill_between(
                result.bin_edges,
                np.r_[lower, lower[-1]],
                np.r_[upper, upper[-1]],
                step="post",
                color=config["color"],
                alpha=0.12,
                linewidth=0.0,
            )

            valid = (
                np.isfinite(sample_events[observable].to_numpy())
                & np.isfinite(sample_events[f"h_{slug}"].to_numpy())
                & np.isfinite(sample_events["weight"].to_numpy())
            )
            inclusive_direct = 4.0 * np.pi * np.sum(
                sample_events.loc[valid, "weight"].to_numpy()
                * sample_events.loc[valid, f"h_{slug}"].to_numpy()
            )
            np.testing.assert_allclose(
                result.values.sum(),
                inclusive_direct,
                rtol=2.0e-12,
                atol=2.0e-12 * max(1.0, np.abs(inclusive_direct)),
            )

        axis.axhline(0.0, color="black", linewidth=0.8)
        axis.set_xlabel(OBSERVABLE_LABELS[observable])
        axis.set_ylabel(rf"$S_{{{coefficient_key}}}$ per bin")
        axis.set_title(rf"$S_{{{coefficient_key}}}$ versus {OBSERVABLE_LABELS[observable]}")
        if row_index == 0 and column_index == 0:
            axis.legend()

plt.show()


## 7. Density-ratio estimation for both selected coefficients

The three complete samples remain in memory, so no LHE file is reloaded. The neural-network demonstration is trained on `TRAINING_SAMPLE`, configurable below, for both $S_{00;20}$ and $S_{20;20}$.

For either real harmonic component $h(\Omega)$, a rigorous bound $M$ defines

$$t(\Omega)=\frac12\left(1+\frac{h(\Omega)}{M}\right).$$

Every source event is duplicated into a row with signed weight $w_it_i$ and a row with signed weight $w_i(1-t_i)$. The class sums $Z_t$ and $Z_{1-t}$ are retained so that the normalization applied by `density_ratio_trainer` can be undone at inference. Negative weights are never rejected, discarded, or replaced by absolute values.

The source events are split before duplication into fit, member-validation, and final-evaluation samples. Each ensemble member is evaluated using the same signed validation objective used in training. A high-loss or non-finite attempt is retrained with a new seed; after the retry budget is exhausted, only that member is discarded and the notebook continues with the remaining validated members.


In [ ]:
from nsbi_common_utils.training import density_ratio_trainer, predict_with_model

TRAINING_SAMPLE = "signal_background_interference"
TRAINING_COEFFICIENTS = tuple(COEFFICIENTS)
FEATURES = ["m_Z1", "m_Z2", "m_ZZ", "cos_theta_star"]
MODEL_VALIDATION_FRACTION = 0.15
EVALUATION_FRACTION = 0.25
ENSEMBLE_SIZE = 4
MAX_MEMBER_RETRIES = 3
LOSS_MAD_SCALE = 5.0
LOSS_MIN_RELATIVE_EXCESS = 0.05
RUN_MODEL = True
LOAD_TRAINED_MODEL = False

MODELS_ROOT = PROJECT_ROOT / "models" / "angular_ratio"
FIGURES_ROOT = PROJECT_ROOT / "figures" / "angular_ratio"
MODELS_ROOT.mkdir(parents=True, exist_ok=True)
FIGURES_ROOT.mkdir(parents=True, exist_ok=True)


def prepare_coefficient_training(coefficient_key, coefficient_index):
    slug = coefficient_slug(coefficient_key)
    source_events = samples[TRAINING_SAMPLE]
    columns = [*FEATURES, f"h_{slug}", f"t_{slug}", "weight"]
    finite = np.isfinite(source_events[columns].to_numpy(dtype=np.float64)).all(axis=1)
    removed = np.count_nonzero(~finite)
    if removed:
        warnings.warn(
            f"{coefficient_key}: masking {removed}/{len(source_events)} non-finite events; "
            "positive and negative finite weights are all retained.",
            RuntimeWarning,
        )
    coefficient_events = source_events.loc[finite].copy()
    fit_events, held_out_events = train_test_split(
        coefficient_events,
        test_size=MODEL_VALIDATION_FRACTION + EVALUATION_FRACTION,
        random_state=RNG_SEED + 100 * coefficient_index,
    )
    validation_events, evaluation_events = train_test_split(
        held_out_events,
        test_size=EVALUATION_FRACTION / (MODEL_VALIDATION_FRACTION + EVALUATION_FRACTION),
        random_state=RNG_SEED + 100 * coefficient_index + 1,
    )
    fit_events = as_float32_features(fit_events.reset_index(drop=True), FEATURES)
    validation_events = as_float32_features(validation_events.reset_index(drop=True), FEATURES)
    evaluation_events = as_float32_features(evaluation_events.reset_index(drop=True), FEATURES)
    training_dataframe, normalizations = prepare_weighted_classification(
        fit_events,
        fit_events[f"t_{slug}"].to_numpy(),
        shuffle_seed=RNG_SEED + 100 * coefficient_index,
    )
    return {
        "coefficient_key": coefficient_key,
        "slug": slug,
        "source_events": source_events,
        "fit_events": fit_events,
        "validation_events": validation_events,
        "evaluation_events": evaluation_events,
        "training_dataframe": training_dataframe,
        "normalizations": normalizations,
        "bound": coefficient_metadata[coefficient_key]["bound"],
    }


def train_coefficient_ensemble(context, coefficient_index):
    coefficient_key = context["coefficient_key"]
    slug = context["slug"]
    model_dir = MODELS_ROOT / slug
    figure_dir = FIGURES_ROOT / slug
    model_dir.mkdir(parents=True, exist_ok=True)
    figure_dir.mkdir(parents=True, exist_ok=True)

    def make_trainer():
        frame = context["training_dataframe"]
        return density_ratio_trainer(
            dataset=frame,
            weights=frame["weights_normed"].to_numpy(),
            training_labels=frame["train_labels"].to_numpy(),
            features=FEATURES,
            features_scaling=FEATURES,
            sample_name=["t-weighted", "one-minus-t-weighted"],
            output_name=f"angular_moment_{slug}",
            path_to_figures=f"{figure_dir}/",
            path_to_models=f"{model_dir}/",
            use_log_loss=False,
        )

    def train_member(member_index, attempt, load_existing=False):
        seed = RNG_SEED + 1_000_000 * coefficient_index + 10_000 * member_index + attempt
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        trainer = make_trainer()
        try:
            trainer.train(
                hidden_layers=5,
                neurons=1024,
                number_of_epochs=50,
                batch_size=1024,
                learning_rate=1e-3,
                scalerType="MinMax",
                ensemble_index=member_index,
                verbose=1,
                rnd_seed=seed,
                holdout_split=0.25,
                validation_split=0.2,
                callback_patience=10,
                num_workers=4,
                load_trained_models=load_existing,
                calibration=False,
                type_of_calibration="histogram",
                recalibrate_output=True,
                num_bins_cal=100,
            )
            scores = predict_with_model(
                context["validation_events"][FEATURES],
                scaler=trainer.scaler,
                model=trainer.model_NN,
                calibration_model=None,
                use_log_loss=trainer.use_log_loss,
            )
            loss = class_balanced_validation_bce(
                scores,
                context["validation_events"][f"t_{slug}"].to_numpy(),
                context["validation_events"]["weight"].to_numpy(),
            )
            if not np.isfinite(loss):
                raise ValueError(f"non-finite validation loss {loss}")
        except Exception as error:
            failure = f"{type(error).__name__}: {error}"[:500]
            warnings.warn(
                f"{coefficient_key}, member {member_index}, attempt {attempt}: {failure}; retrying when possible.",
                RuntimeWarning,
            )
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            return None, np.inf, seed, failure
        return trainer, loss, seed, None

    trainers = [None] * ENSEMBLE_SIZE
    losses = np.full(ENSEMBLE_SIZE, np.nan)
    attempts = np.zeros(ENSEMBLE_SIZE, dtype=int)
    discarded = np.zeros(ENSEMBLE_SIZE, dtype=bool)
    audit = []
    current_rows = np.full(ENSEMBLE_SIZE, -1, dtype=int)
    for member_index in range(ENSEMBLE_SIZE):
        trainer, loss, seed, failure = train_member(
            member_index, 0, load_existing=LOAD_TRAINED_MODEL
        )
        trainers[member_index] = trainer
        losses[member_index] = loss
        audit.append(dict(member=member_index, attempt=0, seed=seed, validation_bce=loss, failure=failure, status="pending"))
        current_rows[member_index] = len(audit) - 1

    while np.any(~discarded):
        active = np.flatnonzero(~discarded)
        rejected_active, threshold = validation_loss_outlier_mask(
            losses[active],
            mad_scale=LOSS_MAD_SCALE,
            min_relative_excess=LOSS_MIN_RELATIVE_EXCESS,
        )
        rejected = active[rejected_active]
        for member_index, is_rejected in zip(active, rejected_active):
            audit[current_rows[member_index]]["acceptance_threshold"] = threshold
            audit[current_rows[member_index]]["status"] = "rejected" if is_rejected else "accepted"
        if len(rejected) == 0:
            break
        for member_index in rejected:
            if attempts[member_index] >= MAX_MEMBER_RETRIES:
                discarded[member_index] = True
                trainers[member_index] = None
                audit[current_rows[member_index]]["status"] = "discarded"
                warnings.warn(
                    f"{coefficient_key}: discarding member {member_index} after {MAX_MEMBER_RETRIES} retries.",
                    RuntimeWarning,
                )
                continue
            attempts[member_index] += 1
            attempt = int(attempts[member_index])
            trainer, loss, seed, failure = train_member(member_index, attempt)
            trainers[member_index] = trainer
            losses[member_index] = loss
            audit.append(dict(member=member_index, attempt=attempt, seed=seed, validation_bce=loss, failure=failure, status="pending"))
            current_rows[member_index] = len(audit) - 1

    validated_indices = [
        index for index in range(ENSEMBLE_SIZE)
        if not discarded[index] and trainers[index] is not None and np.isfinite(losses[index])
    ]
    validated_trainers = [trainers[index] for index in validated_indices]
    if not validated_trainers:
        warnings.warn(f"{coefficient_key}: no validated ensemble member remains.", RuntimeWarning)
    return {
        **context,
        "trainers": trainers,
        "validated_indices": validated_indices,
        "validated_trainers": validated_trainers,
        "validation_losses": losses,
        "validation_threshold": threshold,
        "validation_audit": pd.DataFrame(audit),
    }


training_results = {}
if RUN_MODEL:
    for coefficient_index, coefficient_key in enumerate(TRAINING_COEFFICIENTS):
        context = prepare_coefficient_training(coefficient_key, coefficient_index)
        training_results[coefficient_key] = train_coefficient_ensemble(context, coefficient_index)
        display(training_results[coefficient_key]["validation_audit"])
else:
    print("Training is disabled. Set RUN_MODEL=True and rerun this cell.")


In [ ]:
for coefficient_key, result in training_results.items():
    if not result["validated_trainers"]:
        continue
    trainer = result["validated_trainers"][0]
    member_index = result["validated_indices"][0]
    trainer.make_calib_plots(observable="score", nbins=50, ensemble_index=member_index)

    audit = result["validation_audit"]
    finite_audit = audit[np.isfinite(audit["validation_bce"])]
    fig, axis = plt.subplots(figsize=(7.5, 4.5))
    colors = finite_audit["status"].map({"accepted": "C0", "rejected": "C3", "discarded": "0.4"})
    axis.scatter(
        finite_audit["member"] + 0.08 * finite_audit["attempt"],
        finite_audit["validation_bce"],
        c=colors,
        s=55,
    )
    threshold = result["validation_threshold"]
    if np.isfinite(threshold):
        axis.axhline(threshold, color="black", linestyle="--", label="acceptance threshold")
        axis.legend()
    axis.set(
        xlabel="ensemble member (retries offset)",
        ylabel="signed validation objective",
        title=rf"$S_{{{coefficient_key}}}$ ensemble validation",
        xticks=np.arange(ENSEMBLE_SIZE),
    )
    fig.tight_layout()


In [ ]:
for coefficient_key, result in training_results.items():
    if result["validated_trainers"]:
        result["validated_trainers"][0].make_reweighted_plots(FEATURES, "linear", 50)


## 8. Recover both conditional moments and close the differential coefficients

For each coefficient, the toolkit score gives the ratio of the two independently normalized signed training measures. The stored factor $Z_t/Z_{1-t}$ restores the unnormalized ratio before the conditional moment is reconstructed. Density ratios are averaged across the validated ensemble members, following the toolkit convention.

The final closure uses only the untouched event-level evaluation split for each coefficient. Both $m_{ZZ}$ and $\cos\theta^*$ are shown, and the direct points carry the signed-weight MC uncertainty from the sum of squared event contributions.


In [ ]:
prediction_results = {}
if RUN_MODEL:
    for coefficient_key, result in training_results.items():
        trainers = result["validated_trainers"]
        if not trainers:
            continue
        evaluation_events = result["evaluation_events"].copy()
        member_scores = np.stack([
            np.clip(
                np.asarray(
                    predict_with_model(
                        evaluation_events[FEATURES],
                        scaler=trainer.scaler,
                        model=trainer.model_NN,
                        calibration_model=None,
                        use_log_loss=trainer.use_log_loss,
                    )
                ),
                1.0e-8,
                1.0 - 1.0e-8,
            )
            for trainer in trainers
        ])
        member_ratios = member_scores / (1.0 - member_scores)
        normalized_ratio = np.mean(member_ratios, axis=0)
        eta_hat, h_hat = recover_conditional_moment(
            normalized_ratio,
            result["normalizations"],
            result["bound"],
        )
        evaluation_events["eta_hat"] = eta_hat
        evaluation_events["h_hat"] = h_hat
        prediction_results[coefficient_key] = {**result, "evaluation_events": evaluation_events}
        display(
            evaluation_events[
                [*FEATURES, f"h_{result['slug']}", "h_hat"]
            ].head()
        )
else:
    print("Run the training cell to build the two coefficient estimators.")


In [ ]:
if prediction_results:
    fig, axes = plt.subplots(
        len(TRAINING_COEFFICIENTS),
        len(DIFFERENTIAL_BINS),
        figsize=(15, 10),
        constrained_layout=True,
        squeeze=False,
    )
    for row_index, coefficient_key in enumerate(TRAINING_COEFFICIENTS):
        if coefficient_key not in prediction_results:
            continue
        result = prediction_results[coefficient_key]
        evaluation_events = result["evaluation_events"]
        slug = result["slug"]
        # The common factor restores the full-sample normalization in expectation;
        # it never changes the sign of an event weight.
        evaluation_weights = (
            evaluation_events["weight"].to_numpy()
            * len(result["source_events"])
            / len(evaluation_events)
        )
        for column_index, (observable, edges) in enumerate(DIFFERENTIAL_BINS.items()):
            axis = axes[row_index, column_index]
            direct = binned_angular_coefficient(
                evaluation_events[observable].to_numpy(),
                evaluation_events[f"h_{slug}"].to_numpy(),
                evaluation_weights,
                edges,
                fold_flow=True,
            )
            learned = binned_angular_coefficient(
                evaluation_events[observable].to_numpy(),
                evaluation_events["h_hat"].to_numpy(),
                evaluation_weights,
                edges,
                fold_flow=True,
            )
            centers = 0.5 * (edges[1:] + edges[:-1])
            axis.errorbar(
                centers,
                direct.values,
                yerr=direct.uncertainties,
                fmt="o",
                label="direct angular projection",
            )
            axis.stairs(
                learned.values,
                edges,
                linewidth=2,
                label="density-ratio conditional moment",
            )
            axis.axhline(0.0, color="black", linewidth=0.8)
            axis.set(
                xlabel=OBSERVABLE_LABELS[observable],
                ylabel=rf"$S_{{{coefficient_key}}}$ per bin",
                title=rf"$S_{{{coefficient_key}}}$ closure",
            )
            if row_index == 0 and column_index == 0:
                axis.legend()
    plt.show()
else:
    print("No validated ensemble prediction is available for closure.")


### Required validation before physics use

The three-sample inclusive and direct differential results use every finite event and retain all signed weights. The neural estimators should still be checked across architectures and seeds, and the negative-weight fraction and signed-objective stability should be documented. Angle signs should be validated on hand-checked phase-space points. For a final result, event-grouped cross-validation or independently generated samples should be used, and finite-MC, training, and model-ensemble uncertainties should all be propagated. The direct uncertainty bands shown here contain the weighted-MC contribution from $\sum_i(4\pi w_i h_i)^2$; the learned curves do not yet include model uncertainty.
